# 02 - Delta Lake

Este notebook demuestra capacidades de Delta Lake sobre una tabla controlada derivada de `silver.accidentes`.

Objetivos:

- simular una carga incremental mediante `MERGE INTO`;
- demostrar un upsert: actualizar un accidente existente e insertar uno nuevo;
- inspeccionar el historial con `DESCRIBE HISTORY`;
- consultar una versión anterior mediante Time Travel;
- ejecutar `OPTIMIZE` y justificar su uso.

La tabla `silver.accidentes` no se modifica. Todas las pruebas se realizan sobre `silver.accidentes_delta_demo`.

## 1. Configuración

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catalog del proyecto")

PROY_CATALOG = dbutils.widgets.get("catalog_name")
PROY_SCHEMA_SILVER = "silver"
SOURCE_TABLE = f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes"
DEMO_TABLE = f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes_delta_demo"

print(f"Fuente: {SOURCE_TABLE}")
print(f"Demo:   {DEMO_TABLE}")

## 2. Crear una copia controlada de Silver

La tabla demo se recrea en cada ejecución para que el notebook sea reproducible y no modifique la tabla Silver original.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {DEMO_TABLE}")

spark.sql(f"""
    CREATE TABLE {DEMO_TABLE}
    USING DELTA
    AS
    SELECT * FROM {SOURCE_TABLE}
""")

source_count = spark.table(SOURCE_TABLE).count()
demo_count = spark.table(DEMO_TABLE).count()

print(f"Silver original : {source_count:,}")
print(f"Tabla demo      : {demo_count:,}")
print(f"Conteos iguales : {source_count == demo_count}")

## 3. Capturar la versión inicial

Cada escritura sobre una tabla Delta genera una versión en el transaction log. Guardamos la versión previa al `MERGE` para consultarla después mediante Time Travel.

In [ ]:
history_initial = spark.sql(f"DESCRIBE HISTORY {DEMO_TABLE}")

display(history_initial.select("version", "timestamp", "operation").orderBy(F.desc("version")))

initial_version = history_initial.agg(F.max("version").alias("version")).first()["version"]
print(f"Versión inicial: {initial_version}")

## 4. Simular una carga incremental

Se construyen dos registros a partir de una fila real: uno conserva un `croquis` existente para demostrar un UPDATE y otro usa un `croquis` nuevo para demostrar un INSERT.

In [ ]:
df_demo = spark.table(DEMO_TABLE)

croquis_update = df_demo.select("croquis").orderBy("croquis").first()["croquis"]
original_localidad = (
    df_demo.filter(F.col("croquis") == croquis_update)
    .select("localidad")
    .first()["localidad"]
)

NEW_CROQUIS = "DELTA_DEMO_0001"

base_row = df_demo.filter(F.col("croquis") == croquis_update).limit(1)

update_row = (
    base_row
    .withColumn("localidad", F.lit("LOCALIDAD_DEMO_ACTUALIZADA"))
    .withColumn("_source_file", F.lit("delta_demo_incremental"))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

insert_row = (
    base_row
    .withColumn("croquis", F.lit(NEW_CROQUIS))
    .withColumn("localidad", F.lit("LOCALIDAD_DEMO_NUEVA"))
    .withColumn("_source_file", F.lit("delta_demo_incremental"))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

df_incremental = update_row.unionByName(insert_row)
df_incremental.createOrReplaceTempView("actualizaciones_accidentes_demo")

print(f"Croquis a actualizar: {croquis_update}")
print(f"Localidad original:    {original_localidad}")
print(f"Croquis a insertar:    {NEW_CROQUIS}")
display(df_incremental.select("croquis", "fecha_accidente", "gravedad", "clase_accidente", "localidad"))

## 5. Ejecutar `MERGE INTO`

`WHEN MATCHED` actualiza el accidente existente y `WHEN NOT MATCHED` inserta el nuevo registro. Esto representa un patrón de upsert incremental.

In [ ]:
spark.sql(f"""
    MERGE INTO {DEMO_TABLE} AS target
    USING actualizaciones_accidentes_demo AS source
    ON target.croquis = source.croquis
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print("[OK] MERGE ejecutado")

## 6. Validar el upsert

In [ ]:
after_merge_count = spark.table(DEMO_TABLE).count()

print(f"Registros antes : {demo_count:,}")
print(f"Registros después: {after_merge_count:,}")
print(f"Diferencia        : {after_merge_count - demo_count}")

display(
    spark.table(DEMO_TABLE)
    .filter(F.col("croquis").isin(croquis_update, NEW_CROQUIS))
    .select("croquis", "fecha_accidente", "gravedad", "clase_accidente", "localidad", "_source_file")
)

## 7. Revisar el historial

Después del `MERGE` debe aparecer una nueva versión con operación `MERGE`.

In [ ]:
history_after_merge = spark.sql(f"DESCRIBE HISTORY {DEMO_TABLE}")
display(history_after_merge.select("version", "timestamp", "operation", "operationMetrics").orderBy(F.desc("version")))

merge_version = (
    history_after_merge
    .filter(F.col("operation") == "MERGE")
    .agg(F.max("version").alias("version"))
    .first()["version"]
)

print(f"Versión inicial: {initial_version}")
print(f"Versión MERGE  : {merge_version}")

## 8. Time Travel

Se consulta la versión anterior al `MERGE` y se compara con el estado actual de la tabla.

In [ ]:
before_merge = spark.sql(f"""
    SELECT croquis, localidad
    FROM {DEMO_TABLE} VERSION AS OF {initial_version}
    WHERE croquis IN ('{croquis_update}', '{NEW_CROQUIS}')
""")

after_merge = (
    spark.table(DEMO_TABLE)
    .filter(F.col("croquis").isin(croquis_update, NEW_CROQUIS))
    .select("croquis", "localidad")
)

print("ANTES DEL MERGE")
display(before_merge)

print("DESPUÉS DEL MERGE")
display(after_merge)

### Validación explícita del Time Travel

In [ ]:
old_existing = spark.sql(f"""
    SELECT localidad
    FROM {DEMO_TABLE} VERSION AS OF {initial_version}
    WHERE croquis = '{croquis_update}'
""").first()

old_new_count = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM {DEMO_TABLE} VERSION AS OF {initial_version}
    WHERE croquis = '{NEW_CROQUIS}'
""").first()["total"]

current_existing = (
    spark.table(DEMO_TABLE)
    .filter(F.col("croquis") == croquis_update)
    .select("localidad")
    .first()["localidad"]
)

print(f"Localidad antes : {old_existing['localidad']}")
print(f"Localidad ahora : {current_existing}")
print(f"Nuevo croquis en versión anterior: {old_new_count}")

## 9. `OPTIMIZE`

Las cargas incrementales frecuentes pueden generar muchos archivos pequeños. `OPTIMIZE` reorganiza físicamente los archivos de una tabla Delta para reducir esa fragmentación.

En este proyecto la tabla tiene solo unas decenas de miles de filas, por lo que no se espera una mejora material de rendimiento. Se ejecuta para demostrar la operación y documentar el escenario en el que sería útil: tablas de mayor volumen con escrituras incrementales frecuentes.

In [ ]:
optimize_result = spark.sql(f"OPTIMIZE {DEMO_TABLE}")
display(optimize_result)

## 10. Historial final

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {DEMO_TABLE}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.desc("version"))
)

## 11. Validaciones finales

In [ ]:
final_count = spark.table(DEMO_TABLE).count()

existing_updated_ok = (
    spark.table(DEMO_TABLE)
    .filter((F.col("croquis") == croquis_update) & (F.col("localidad") == "LOCALIDAD_DEMO_ACTUALIZADA"))
    .count() == 1
)

new_inserted_ok = (
    spark.table(DEMO_TABLE)
    .filter(F.col("croquis") == NEW_CROQUIS)
    .count() == 1
)

count_ok = final_count == demo_count + 1

print(f"UPDATE correcto      : {existing_updated_ok}")
print(f"INSERT correcto      : {new_inserted_ok}")
print(f"Conteo final correcto: {count_ok}")
print(f"Filas finales        : {final_count:,}")

## Conclusiones

En esta demostración se creó una copia controlada de `silver.accidentes`, se simuló una carga incremental, `MERGE INTO` realizó un UPDATE y un INSERT, `DESCRIBE HISTORY` permitió inspeccionar las versiones, Time Travel mostró el estado anterior y `OPTIMIZE` demostró la compactación física disponible en Delta Lake.